# Supply Chain Analytics — Data Exploration, KPIs & Root-Cause Analysis

**Tools:** SQL (SQLite/PostgreSQL-compatible) · Python (pandas, matplotlib, seaborn) · Power BI

This notebook analyzes 100+ supply chain records covering inventory, supplier performance,
warehouse operations, and delivery efficiency. It builds the same cleaned star-schema data
used by the SQL scripts in `/sql` and the Power BI model in `/powerbi`, then walks through
KPI tracking (Order Fill Rate, Inventory Turnover, On-Time Delivery, Warehouse Utilization)
and root-cause analysis of operational bottlenecks.

**Project structure**
```
supply-chain-analytics/
├── data/            raw + cleaned/star-schema data
├── sql/             schema + KPI/analysis queries
├── python/          this notebook + standalone scripts
├── powerbi/         Power BI data model, DAX measures, build guide
├── visuals/         exported chart images
└── reports/         written insights & recommendations
```


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 50)


## 1. Load & Inspect Raw Data

In [ ]:
df_raw = pd.read_csv("../data/raw/supply_chain_data.csv")
print(df_raw.shape)
df_raw.head()


In [ ]:
df_raw.info()


In [ ]:
print("Nulls per column:")
print(df_raw.isnull().sum())
print("\nDuplicate rows:", df_raw.duplicated().sum())


## 2. Clean Data & Engineer KPI Fields

This mirrors `python/01_data_cleaning.py`. See that script for the full, reusable version —
here we inline it so the notebook is self-contained.


In [ ]:
df = df_raw.copy()
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))
df.insert(0, "record_id", range(1, len(df) + 1))

# Order Fill Rate (%) - capped at 100%
df["order_fill_rate"] = np.where(
    df["order_quantities"] > 0,
    (df["number_of_products_sold"] / df["order_quantities"]).clip(upper=1) * 100,
    np.nan,
)

# Inventory Turnover
df["inventory_turnover"] = np.where(
    df["stock_levels"] > 0, df["number_of_products_sold"] / df["stock_levels"], np.nan
)

# On-Time Delivery flag (shipping time <= median for its mode)
median_by_mode = df.groupby("transportation_modes")["shipping_times"].transform("median")
df["on_time_delivery_flag"] = (df["shipping_times"] <= median_by_mode).astype(int)

# Warehouse Utilization (%)
denom = df["stock_levels"] + df["order_quantities"]
df["warehouse_utilization_pct"] = np.where(denom > 0, (df["stock_levels"] / denom) * 100, np.nan)

# Profitability
df["profit"] = df["revenue_generated"] - df["costs"]
df["profit_margin_pct"] = np.where(df["revenue_generated"] > 0,
                                     (df["profit"] / df["revenue_generated"]) * 100, np.nan)

df["total_shipping_cost"] = df["number_of_products_sold"] * df["shipping_costs"]
df["quality_pass_flag"] = (df["inspection_results"].str.lower() == "pass").astype(int)

df.to_csv("../data/processed/supply_chain_cleaned.csv", index=False)
df.head()


## 3. Headline KPIs

In [ ]:
kpis = {
    "Total Revenue ($)": df["revenue_generated"].sum(),
    "Total Profit ($)": df["profit"].sum(),
    "Avg Profit Margin (%)": df["profit_margin_pct"].mean(),
    "Order Fill Rate (%)": df["order_fill_rate"].mean(),
    "Inventory Turnover (x)": df["inventory_turnover"].mean(),
    "On-Time Delivery (%)": df["on_time_delivery_flag"].mean() * 100,
    "Avg Warehouse Utilization (%)": df["warehouse_utilization_pct"].mean(),
}
for k, v in kpis.items():
    print(f"{k:32s}: {v:,.2f}")


In [ ]:
pct_kpis = pd.Series({
    "Order Fill Rate (%)": kpis["Order Fill Rate (%)"],
    "On-Time Delivery (%)": kpis["On-Time Delivery (%)"],
    "Warehouse Utilization (%)": kpis["Avg Warehouse Utilization (%)"],
})
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), gridspec_kw={"width_ratios": [2, 1]})
sns.barplot(x=pct_kpis.index, y=pct_kpis.values, hue=pct_kpis.index, palette="mako", legend=False, ax=axes[0])
axes[0].set_title("Headline KPIs (%)"); axes[0].set_ylabel("Percent"); axes[0].set_ylim(0, 100)
axes[0].tick_params(axis="x", rotation=15)
axes[1].bar(["Inventory\nTurnover"], [kpis["Inventory Turnover (x)"]], color="#4c72b0")
axes[1].set_title("Avg Inventory Turnover")
plt.tight_layout()
plt.show()


## 4. Revenue Analysis

In [ ]:
rev = df.groupby("product_type")["revenue_generated"].sum().sort_values(ascending=False)
plt.figure(figsize=(7, 4.5))
sns.barplot(x=rev.index, y=rev.values, hue=rev.index, palette="viridis", legend=False)
plt.title("Revenue Generated by Product Type"); plt.ylabel("Total Revenue ($)"); plt.xlabel("")
plt.tight_layout(); plt.show()


In [ ]:
rev_loc = df.groupby("location")["revenue_generated"].sum().sort_values(ascending=False)
plt.figure(figsize=(6.5, 6.5))
plt.pie(rev_loc.values, labels=rev_loc.index, autopct="%1.1f%%", colors=sns.color_palette("viridis", len(rev_loc)))
plt.title("Revenue Distribution by Location")
plt.tight_layout(); plt.show()


## 5. Cost, Price & Profitability

In [ ]:
price_cost = df.groupby("product_type").agg(Price=("price", "sum"),
                                             Manufacturing_Cost=("manufacturing_costs", "sum")).reset_index()
price_cost_m = price_cost.melt(id_vars="product_type", var_name="Metric", value_name="Value")
plt.figure(figsize=(7.5, 4.5))
sns.barplot(data=price_cost_m, x="product_type", y="Value", hue="Metric", palette="Set2")
plt.title("Price vs. Manufacturing Cost by Product Type"); plt.xlabel("")
plt.tight_layout(); plt.show()


In [ ]:
profit = df.groupby("product_type")["profit"].sum().sort_values()
plt.figure(figsize=(7, 4.5))
colors = ["#d62728" if v < 0 else "#2ca02c" for v in profit.values]
plt.barh(profit.index, profit.values, color=colors)
plt.title("Overall Profitability by Product Type"); plt.xlabel("Profit ($)")
plt.tight_layout(); plt.show()


## 6. Quality & Supplier Performance

In [ ]:
plt.figure(figsize=(6.5, 4.5))
sns.boxplot(data=df, x="inspection_results", y="defect_rates", hue="inspection_results",
            palette="coolwarm", legend=False)
plt.title("Defect Rates by Inspection Result"); plt.xlabel("Inspection Result"); plt.ylabel("Defect Rate (%)")
plt.tight_layout(); plt.show()


In [ ]:
supplier_perf = df.groupby("supplier_name").agg(avg_defect_rate=("defect_rates", "mean"),
                                                  avg_lead_time=("lead_times", "mean")).reset_index()
plt.figure(figsize=(7, 5))
sns.scatterplot(data=supplier_perf, x="avg_lead_time", y="avg_defect_rate",
                 hue="supplier_name", s=160, palette="tab10")
plt.title("Supplier Performance: Lead Time vs. Defect Rate")
plt.xlabel("Avg Lead Time (days)"); plt.ylabel("Avg Defect Rate (%)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()


## 7. Transportation & Shipping

In [ ]:
mode_summary = df.groupby("transportation_modes").agg(
    avg_shipping_cost=("shipping_costs", "mean"),
    avg_shipping_time=("shipping_times", "mean"),
    n=("record_id", "count")).reset_index()
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sns.barplot(data=mode_summary, x="transportation_modes", y="n", hue="transportation_modes",
            palette="crest", legend=False, ax=axes[0])
axes[0].set_title("Shipment Count by Mode"); axes[0].set_xlabel("")
sns.scatterplot(data=mode_summary, x="avg_shipping_time", y="avg_shipping_cost",
                 hue="transportation_modes", s=180, palette="crest", ax=axes[1])
axes[1].set_title("Avg Shipping Time vs. Cost")
plt.tight_layout(); plt.show()


## 8. Inventory & Warehouse Utilization

In [ ]:
wh = df.groupby("location")["warehouse_utilization_pct"].mean().sort_values(ascending=False)
plt.figure(figsize=(7, 4.5))
sns.barplot(x=wh.index, y=wh.values, hue=wh.index, palette="flare", legend=False)
plt.title("Average Warehouse Utilization by Location"); plt.ylabel("Warehouse Utilization (%)"); plt.xlabel("")
plt.tight_layout(); plt.show()


## 9. Correlation Overview

In [ ]:
numeric_cols = ["price", "availability", "number_of_products_sold", "revenue_generated",
                "stock_levels", "lead_times", "order_quantities", "shipping_times",
                "shipping_costs", "manufacturing_costs", "defect_rates", "costs", "profit"]
corr = df[numeric_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="RdBu_r", center=0, square=True)
plt.title("Correlation Heatmap: Core Supply Chain Metrics")
plt.tight_layout(); plt.show()


## 10. Root-Cause Analysis: Where Are the Bottlenecks?

Mirrors `sql/06_root_cause_analysis.sql`.

In [ ]:
bins = [0, 10, 20, df["manufacturing_lead_time"].max() + 1]
labels = ["0-10 days", "11-20 days", "21+ days"]
df["lead_time_bucket"] = pd.cut(df["manufacturing_lead_time"], bins=bins, labels=labels)
lt_defect = df.groupby("lead_time_bucket", observed=True)["defect_rates"].mean().round(2)
print("Avg defect rate by manufacturing lead-time bucket:")
print(lt_defect)


In [ ]:
bottlenecks = df.nsmallest(10, "order_fill_rate")[
    ["sku", "product_type", "order_quantities", "number_of_products_sold", "order_fill_rate"]
]
print("Lowest Order Fill Rate SKUs (bottleneck candidates):")
bottlenecks


In [ ]:
route_perf = df.groupby("routes").agg(avg_lead_time=("lead_times", "mean"),
                                        avg_cost=("costs", "mean"),
                                        avg_defect_rate=("defect_rates", "mean")).round(2)
route_perf.sort_values("avg_lead_time", ascending=False)


## 11. Key Takeaways

- **On-Time Delivery (61%)** is the weakest headline KPI — see `reports/insights_and_recommendations.md`
  for the full write-up and recommended actions.
- **Skincare** is the strongest product line by revenue, profit, and inventory turnover.
- **Route B** has the highest average lead time and cost — a strong candidate for logistics review.
- Longer manufacturing lead times (21+ days) correlate with higher defect rates, suggesting
  quality control should be reinforced for longer production runs.

For the interactive KPI dashboard, see the Power BI build guide in `/powerbi/README.md`
(data model + DAX measures are ready to import — see `data/processed/`).
